# 10. Обучение трансформера

**Цель:** Реализовать полный цикл обучения: функцию потерь, оптимизатор, планировщик LR, мониторинг, сохранение модели и эксперименты с гиперпараметрами.

---

In [1]:
# Настройка: logging, torch, numpy, matplotlib — стандартные импорты для обучения
# Выбор устройства: MPS (Apple GPU) > CUDA > CPU
import sys, os, logging, math
LOG_LEVEL = os.getenv("LOG_LEVEL", "DEBUG")
logging.basicConfig(level=getattr(logging, LOG_LEVEL), format="%(asctime)s [%(levelname)s] %(name)s: %(message)s", stream=sys.stderr)
log = logging.getLogger("training")

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
log.info("Используем устройство: %s", device)

2026-05-31 19:40:17,835 [DEBUG] matplotlib: matplotlib data path: /Users/demanbudaev/PycharmProjects/Ashat_AI/.venv/lib/python3.9/site-packages/matplotlib/mpl-data
2026-05-31 19:40:17,843 [DEBUG] matplotlib: CONFIGDIR=/Users/demanbudaev/.matplotlib
2026-05-31 19:40:17,850 [DEBUG] matplotlib: interactive is False
2026-05-31 19:40:17,850 [DEBUG] matplotlib: platform is darwin
2026-05-31 19:40:17,884 [DEBUG] matplotlib: CACHEDIR=/Users/demanbudaev/.matplotlib
2026-05-31 19:40:17,886 [DEBUG] matplotlib.font_manager: Using fontManager instance from /Users/demanbudaev/.matplotlib/fontlist-v390.json
2026-05-31 19:40:18,037 [INFO] training: Using device: mps


## 10.1 Функция потерь с игнорированием padding

CrossEntropyLoss с `ignore_index=PAD` — токены PAD не участвуют в подсчёте loss.

In [ ]:
# CrossEntropyLoss с ignore_index=PAD: токены padding исключены из подсчёта loss
# Это предотвращает штрафование модели за предсказание PAD-позиций
BOS, EOS, PAD = 0, 1, 2
criterion = nn.CrossEntropyLoss(ignore_index=PAD)
log.info("Loss: CrossEntropyLoss with ignore_index=PAD")

## 10.2 Noam Scheduler

**Формула (Vaswani et al.):**
$$\text{lr} = d_{model}^{-0.5} \cdot \min(\text{step}^{-0.5}, \text{step} \cdot \text{warmup}^{-1.5})$$

**Фазы:**
1. Linear warmup: LR растёт от 0 до пика за warmup_steps
2. Decay: LR убывает обратно пропорционально sqrt(step)

In [ ]:
# Формула планировщика Noam: lrate = d_model^(-0.5) * min(step^(-0.5), step * warmup^(-1.5))
# Почему разогрев: предотвращает большие обновления градиентов в начале, когда модель случайно инициализирована
#   — случайная инициализация даёт большие/хаотичные градиенты; линейный рост LR стабилизирует раннее обучение
# Почему спад после разогрева: позволяет точную сходимость
#   — вблизи оптимума маленькие шаги позволяют точно сойтись; избегает перелёта
# Ранняя остановка: отслеживает validation loss каждую эпоху; останавливается при плато
# Чекпоинтинг: сохраняет лучшую модель (deepcopy) при улучшении val loss
log.debug("Implementing Noam scheduler")

class NoamScheduler:
    def __init__(self, optimizer, d_model, warmup_steps=4000, factor=1.0):
        self.optimizer = optimizer
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        self.factor = factor
        self._step = 0
        self._rate = 0
    
    def step(self):
        self._step += 1
        self._rate = self.factor * (self.d_model ** -0.5) * min(self._step ** -0.5, self._step * self.warmup_steps ** -1.5)
        for p in self.optimizer.param_groups:
            p['lr'] = self._rate
        self.optimizer.step()
    
    def get_rate(self):
        return self._rate

# Визуализация LR schedule
d_model = 32
optimizer_dummy = torch.optim.Adam([torch.tensor(0.)], lr=0)
scheduler = NoamScheduler(optimizer_dummy, d_model, warmup_steps=20)
lrs = []
for _ in range(200):
    scheduler.step()
    lrs.append(scheduler.get_rate())

plt.figure(figsize=(10, 4))
plt.plot(lrs)
plt.axvline(20, color='red', linestyle='--', alpha=0.5, label='Warmup end')
plt.xlabel('Step')
plt.ylabel('Learning rate')
plt.title(f'Noam Scheduler (d_model={d_model}, warmup=20)')
plt.legend()
plt.grid(True)
plt.show()
log.info("Noam scheduler visualized")

### Noam scheduler: разбор формулы, почему warmup + decay

**Фаза linear warmup** ($\text{step} < \text{warmup\_steps}$):
$$\text{lrate} = d_{model}^{-0.5} \cdot \text{step} \cdot \text{warmup}^{-1.5}$$
LR растёт линейно от 0 до пикового значения на шаге `warmup_steps`.
Это предотвращает взрывные градиенты в начале, когда веса ещё случайны — градиенты хаотичны, и большой LR сразу приведёт к расходимости.

**Фаза inverse square root decay** ($\text{step} \geq \text{warmup\_steps}$):
$$\text{lrate} = d_{model}^{-0.5} \cdot \text{step}^{-0.5}$$
LR убывает как $1/\sqrt{\text{step}}$, обеспечивая мелкие шаги для точной сходимости.
После warmup модель уже в окрестности оптимума — нужны аккуратные корректировки, а не крупные прыжки.

**Итог:** warmup + inverse sqrt decay — стандартный scheduler для трансформеров (Vaswani et al., 2017), заменяющий cosine annealing.

## 10.3 Компоненты трансформера (минимум для обучения)

In [ ]:
# Строительные блоки трансформера: MultiHeadAttention, FeedForward, EncoderBlock
# Pre-LayerNorm архитектура (норма перед подслоями) для стабильности обучения
# Инициализация Xavier для всех матриц весов с dim > 1
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, Q, K, V, mask=None):
        batch = Q.size(0)
        n_heads = self.W_Q.out_features // self.d_k
        Q = self.W_Q(Q).view(batch, -1, n_heads, self.d_k).transpose(1, 2)
        K = self.W_K(K).view(batch, -1, n_heads, self.d_k).transpose(1, 2)
        V = self.W_V(V).view(batch, -1, n_heads, self.d_k).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = self.dropout(F.softmax(scores, dim=-1))
        output = torch.matmul(attn, V).transpose(1, 2).contiguous().view(batch, -1, self.W_O.in_features)
        return self.W_O(output)

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=None, dropout=0.1):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.fc2(self.dropout(F.gelu(self.fc1(x))))

class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
    def forward(self, x, mask=None):
        x = x + self.dropout1(self.attention(self.norm1(x), self.norm1(x), self.norm1(x), mask))
        x = x + self.dropout2(self.ffn(self.norm2(x)))
        return x

class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, num_layers, max_len=100, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).float().unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
        self.dropout_pe = nn.Dropout(dropout)
        self.layers = nn.ModuleList([EncoderBlock(d_model, n_heads, 4*d_model, dropout) for _ in range(num_layers)])
        self.output_proj = nn.Linear(d_model, vocab_size)
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def forward(self, x, mask=None):
        x = self.dropout_pe(self.embedding(x) * math.sqrt(self.d_model) + self.pe[:, :x.size(1), :])
        for layer in self.layers:
            x = layer(x, mask)
        return self.output_proj(x)

log.debug("Transformer model redefined for training notebook")

## 10.4 Цикл обучения с мониторингом

Включает: train/val loss, accuracy, early stopping, сохранение модели.

In [ ]:
# Синтетические данные: случайные последовательности с BOS/EOS/PAD токенами
# AdamW оптимизатор (раздельный weight decay, не L2) для лучшего обобщения
# Планировщик Noam с warmup=50 шагов; d_model=32 определяет базовый масштаб LR
log.debug("Setting up data and training")

vocab_size = 16

def make_data(num_samples, max_len):
    src, tgt = [], []
    for _ in range(num_samples):
        length = np.random.randint(2, max_len + 1)
        seq = np.random.randint(3, vocab_size, size=length).tolist()
        src.append([BOS] + seq + [EOS] + [PAD] * (max_len - length))
        tgt.append(seq + [EOS] + [PAD] * (max_len - length))
    return torch.tensor(src), torch.tensor(tgt)

train_src, train_tgt = make_data(800, 8)
val_src, val_tgt = make_data(200, 8)

model = Transformer(vocab_size=vocab_size, d_model=32, n_heads=4, num_layers=3, max_len=12).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, betas=(0.9, 0.98), eps=1e-9)
scheduler = NoamScheduler(optimizer, d_model=32, warmup_steps=50)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)

print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")
log.info("Training setup complete")

In [ ]:
# Ранняя остановка: отслеживает val loss; если нет улучшения 'patience' эпох,
#   обучение останавливается — предотвращает переобучение
# Чекпоинтинг: сохраняет state dict модели и оптимизатора при улучшении val loss;
#   best_state восстанавливается после обучения (готовый к возобновлению чекпоинт)
# Gradient clipping (norm=1.0) предотвращает взрыв градиентов в глубоких трансформерах
log.info("Starting training loop")

n_epochs = 60
batch_size = 32
best_loss = float('inf')
patience = 10
wait = 0
best_state = None

train_losses, val_losses, lr_history = [], [], []

for epoch in range(n_epochs):
    model.train()
    train_loss = 0
    perm = torch.randperm(len(train_src))
    
    for i in range(0, len(train_src), batch_size):
        idx = perm[i:i+batch_size]
        src = train_src[idx].to(device)
        tgt = train_tgt[idx].to(device)
        
        output = model(src)
        loss = criterion(output[:, :tgt.size(1), :].reshape(-1, vocab_size), tgt.reshape(-1))
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scheduler.step()
        
        train_loss += loss.item()
    
    # Валидация
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for i in range(0, len(val_src), batch_size):
            src = val_src[i:i+batch_size].to(device)
            tgt = val_tgt[i:i+batch_size].to(device)
            output = model(src)
            val_loss += criterion(output[:, :tgt.size(1), :].reshape(-1, vocab_size), tgt.reshape(-1))  # reshape обоих тензоров к (batch*seq, vocab) для CrossEntropy.item()
    
    avg_train = train_loss / (len(train_src) / batch_size)
    avg_val = val_loss / (len(val_src) / batch_size)
    train_losses.append(avg_train.item())
    val_losses.append(avg_val.item())
    lr_history.append(scheduler.get_rate())
    
    if epoch % 10 == 0:
        log.info("Epoch %d: train=%.4f, val=%.4f, lr=%.2e", epoch, avg_train, avg_val, scheduler.get_rate())
    
    # Ранняя остановка
    if avg_val < best_loss:
        best_loss = avg_val
        best_state = deepcopy(model.state_dict())
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            log.info("Early stopping at epoch %d", epoch)
            break

# Загрузка лучшей модели
model.load_state_dict(best_state)
log.info("Training complete. Best val loss: %.4f", best_loss)

### AdamW vs Adam: зачем weight decay

В стандартном Adam L2-регуляризация добавляется в loss:
$$L_{\text{total}} = L_{\text{data}} + \frac{\lambda}{2}\|\theta\|^2$$
Градиент $\lambda\theta$ смешивается с градиентом loss до адаптивной нормализации. Это проблематично: параметры с большими градиентами получают ослабленную регуляризацию из-за адаптивного LR.

**AdamW (Loshchilov & Hutter, 2019):** weight decay **отделён** от градиентного шага:
$$\theta_{t+1} = \theta_t - \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon} - \eta \cdot \lambda \cdot \theta_t$$
Последнее слагаемое $\eta \cdot \lambda \cdot \theta_t$ не зависит от $\hat{m}_t, \hat{v}_t$ — применяет одинаковую силу штрафа ко всем параметрам.

**На практике:** AdamW стабильнее, даёт лучшую обобщающую способность и менее чувствителен к выбору $\lambda$.
Используем `torch.optim.AdamW` с `weight_decay=0.01` (или подбором).

In [ ]:
# Визуализация: кривые train/val loss, расписание LR и финальная точность
# Точность считается только по non-pad токенам для честной метрики
log.debug("Plotting training curves")

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].plot(train_losses, label='Train')
axes[0].plot(val_losses, label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curves')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(lr_history)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Learning rate')
axes[1].set_title('LR Schedule')
axes[1].grid(True)

# Точность (Accuracy)
model.eval()
with torch.no_grad():
    src_sample = val_src[:100].to(device)
    tgt_sample = val_tgt[:100].to(device)
    output = model(src_sample)
    preds = output.argmax(-1)
    non_pad = tgt_sample != PAD
    acc = (preds[non_pad] == tgt_sample[non_pad]).float().mean().item()

axes[2].bar(['Точность'], [acc])
axes[2].set_ylim(0, 1)
axes[2].set_title(f'Validation Accuracy: {acc:.2%}')
axes[2].grid(True, axis='y')

plt.tight_layout()
plt.show()
log.info("Training curves plotted")

## 10.5 Сохранение и загрузка модели

### Overfitting в трансформерах: dropout, weight decay, early stopping

Трансформеры склонны к переобучению из-за огромного числа параметров и слабых inductive biases.
Три уровня защиты, которые мы используем:

**1. Dropout** — применяется после каждого под-слоя:
- Attention dropout: после softmax (`self.dropout` в MultiHeadAttention, $p=0.1$)
- FFN dropout: после GELU активации (`self.dropout` в FeedForward)
- Embedding dropout: `self.dropout_pe` после positional encoding
- Вероятность $p=0.1$ для малых моделей, $p=0.2{-}0.3$ для больших

**2. Weight decay (AdamW)** — штрафует большие веса, предпочитая простые решения:
- Параметры Linear и Embedding получают WD; LayerNorm и biases — исключаются
- Работает ортогонально dropout: dropout — стохастическое зашумление, WD — явная регуляризация нормы

**3. Early stopping** — останавливает обучение при ухудшении val loss:
- `patience`: число эпох без улучшения перед остановкой
- `best_state = deepcopy(state_dict)`: сохраняем лучшую модель
- После цикла загружаем `best_state`, а не последние веса

**Вывод:** комбинация всех трёх методов даёт надёжную защиту от overfitting.

In [ ]:
# Чекпоинт содержит: эпоха, состояние модели, состояние оптимизатора, losses
# Позволяет полностью возобновить обучение: загрузить чекпоинт и продолжить
# Сохраняем лучшую модель (минимальный val loss) — не только последнюю эпоху
log.info("Saving model checkpoint")

os.makedirs('models', exist_ok=True)
checkpoint = {
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_loss': train_losses[-1],
    'val_loss': val_losses[-1],
}
torch.save(checkpoint, 'models/transformer_checkpoint.pt')
print("Model saved to models/transformer_checkpoint.pt")

# Загрузка
loaded = torch.load('models/transformer_checkpoint.pt', map_location=device)
print(f"Loaded checkpoint: epoch={загружен['epoch']}, val_loss={загружен['val_loss']:.4f}")
log.info("Model save/load test passed")

## 10.6 Эксперименты: влияние гиперпараметров

Сравним разные конфигурации: d_model, learning rate, warmup.

In [ ]:
# Абляционное исследование: меняем d_model, n_heads, num_layers с фиксированным warmup
# Быстрое обучение (40 эпох) для сравнения скорости сходимости разных конфигураций
# Столбчатая диаграмма финального loss для каждой конфигурации — определяет лучшую архитектуру
log.debug("Running hyperparameter experiments")

def train_quick(config):
    m = Transformer(vocab_size=vocab_size, **config).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=0.001, betas=(0.9, 0.98))
    sch = NoamScheduler(opt, d_model=config['d_model'], warmup_steps=30)
    
    losses = []
    for _ in range(40):
        perm = torch.randperm(len(train_src))
        epoch_loss = 0
        for i in range(0, len(train_src), 64):
            idx = perm[i:i+64]
            src = train_src[idx].to(device)
            tgt = train_tgt[idx].to(device)
            out = m(src)
            loss = criterion(out.reshape(-1, vocab_size), tgt.reshape(-1))
            opt.zero_grad()
            loss.backward()
            sch.step()
            epoch_loss += loss.item()
        losses.append(epoch_loss / (len(train_src) / 64))
    return losses[-1]

configs = [
    {'d_model': 16, 'n_heads': 2, 'num_layers': 2, 'max_len': 12, 'dropout': 0.1},
    {'d_model': 32, 'n_heads': 4, 'num_layers': 2, 'max_len': 12, 'dropout': 0.1},
    {'d_model': 64, 'n_heads': 4, 'num_layers': 3, 'max_len': 12, 'dropout': 0.1},
    {'d_model': 32, 'n_heads': 4, 'num_layers': 4, 'max_len': 12, 'dropout': 0.1},
]

results = []
for cfg in configs:
    loss = train_quick(cfg)
    label = f"d={cfg['d_model']} h={cfg['n_heads']} L={cfg['num_layers']}"
    results.append((label, loss))
    log.info("Config %s: final loss=%.4f", label, loss)

plt.figure(figsize=(10, 5))
labels, losses = zip(*results)
plt.bar(labels, losses)
plt.xlabel('Configuration')
plt.ylabel('Final loss')
plt.title('Hyperparameter Comparison')
plt.xticks(rotation=15)
plt.grid(True, axis='y')
plt.tight_layout()
plt.show()
log.info("Hyperparameter experiments complete")

In [ ]:
# Итог: все компоненты обучения продемонстрированы в этом ноутбуке
print("=== Training Transformer complete ===")
print("Topics covered:")
print("  - CrossEntropyLoss with padding ignore")
print("  - AdamW optimizer")
print("  - Noam scheduler (warmup + decay)")
print("  - Full training loop with validation")
print("  - Early stopping")
print("  - Model checkpoint save/load")
print("  - Hyperparameter experiments")
print(f"  - Best validation loss: {best_loss:.4f}")
log.info("Training notebook complete")

📚 **Полезные ссылки:**
- [Attention Is All You Need (Vaswani et al., 2017)](https://arxiv.org/abs/1706.03762) — Noam scheduler
- [AdamW: Decoupled Weight Decay Regularization (Loshchilov & Hutter, 2019)](https://arxiv.org/abs/1711.05101)
- [PyTorch: LambdaLR scheduler](https://pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.LambdaLR.html)
